# 03b — (Optionnel) détections DETR pour des métadonnées de requête sans oracle

```text
(OPTIONNEL) Détections d'objets par DETR -> métadonnées « réalistes » côté requête.
PAR DÉFAUT : val + test uniquement (200 images, ~quelques minutes sur CPU). La KB garde les annotations COCO ;
pour détecter aussi la KB (10 000 images, plusieurs heures sur CPU -> prévoir un GPU/Colab) : --splits kb val test.

Pourquoi : dans 03, les objets viennent des annotations COCO (vérité terrain). Pour une image de test, utiliser la
vérité terrain comme requête de filtrage = oracle. Les groupes Metadata-Filtered RAG (G28-G30) doivent utiliser CES
détections pour les requêtes val/test (et peuvent utiliser GT ou détections pour la KB — à documenter).

Sortie : tables/detections_detr.jsonl  {image_id, split, detections:[{category, score, bbox_norm}]}, seuil = config.detector.conf_threshold
NB : nécessite torch + transformers + accès à huggingface.co (non testé hors-ligne dans l'environnement de développement).
```

> Projet : étude comparative de techniques RAG pour le captioning d'images (MS COCO) — **groupe G1**. Exécuter les notebooks dans l'ordre : 00 → 01 → 01b → 02 → … → 09. Les helpers partagés sont dans `common.py`, `visual.py`, `rag_data.py`.

In [1]:
import os
# --- paramètre : "config.yaml" = vraies données COCO ; "config_test.yaml" = mini-COCO synthétique (test sans réseau)
CONFIG = os.environ.get("RAG_CONFIG", "config.yaml")
os.environ["RAG_CONFIG"] = CONFIG
print("config utilisée :", CONFIG)

config utilisée : config.yaml


## Imports et constantes

In [2]:
import argparse
import numpy as np
import pandas as pd
from PIL import Image
from common import get_logger, load_config, mkdirs, paths, write_json, write_jsonl
log = get_logger("detr")

### `main()`

In [3]:
def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--splits", nargs="*", default=["val", "test"], choices=["kb", "val", "test"])
    args = ap.parse_args()
    import torch
    from transformers import AutoImageProcessor, DetrForObjectDetection
    cfg = load_config(); p = paths(cfg); mkdirs(p)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    proc = AutoImageProcessor.from_pretrained(cfg["detector"]["model"])
    model = DetrForObjectDetection.from_pretrained(cfg["detector"]["model"]).to(dev).eval()
    thr = cfg["detector"]["conf_threshold"]
    imgs = pd.read_csv(p["tables"] / "images.csv"); imgs = imgs[(imgs.status == "ok") & imgs.split.isin(args.splits)].sort_values(["split", "image_id"])
    log.info("splits détectés : %s -> %d images", args.splits, len(imgs))
    out = []
    rows = list(imgs.itertuples())
    for i in range(0, len(rows), args.batch):
        b = rows[i:i + args.batch]
        ims = [Image.open(p["art"] / r.path_rgb).convert("RGB") for r in b]
        with torch.no_grad():
            inp = proc(images=ims, return_tensors="pt").to(dev)
            o = model(**inp)
        res = proc.post_process_object_detection(o, threshold=thr, target_sizes=torch.tensor([im.size[::-1] for im in ims]).to(dev))
        for r, im, d in zip(b, ims, res):
            W, H = im.size
            dets = [{"category": model.config.id2label[int(l)], "score": round(float(s), 4),
                     "bbox_norm": [round(float(v), 4) for v in (bx[0] / W, bx[1] / H, bx[2] / W, bx[3] / H)]}
                    for s, l, bx in zip(d["scores"].cpu(), d["labels"].cpu(), d["boxes"].cpu())]
            out.append({"image_id": int(r.image_id), "split": r.split, "detections": dets})
        if (i // args.batch) % 5 == 0:
            log.info("%d/%d", i, len(rows))
    write_jsonl(p["tables"] / "detections_detr.jsonl", out)
    write_json(p["audit"] / "detector_info.json", {"model": cfg["detector"]["model"], "conf_threshold": thr, "n_images": len(out), "splits": args.splits,
                                                  "note": "DETR COCO : ses 91 ids/80 classes sont les mêmes classes que COCO -> comparables aux annotations GT"})

## Lancer l'étape

In [4]:
import sys
sys.argv = ["notebook", "--splits", "val", "test"]          # pas d'arguments CLI dans un notebook
try:
    main()
except SystemExit as e:               # certains scripts terminent par sys.exit(code)
    print("terminé, code de sortie :", e.code)

19:47:37 | INFO | HTTP Request: GET https://huggingface.co/api/models/facebook/detr-resnet-50 "HTTP/1.1 200 OK"
19:47:37 | INFO | HTTP Request: GET https://huggingface.co/api/models/facebook/detr-resnet-50/revision/main "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


19:47:41 | INFO | splits détectés : ['val', 'test'] -> 200 images
19:52:32 | INFO | 0/200
20:05:07 | INFO | 80/200
20:18:18 | INFO | 160/200


## Résultats

In [5]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
from common import load_config, paths
cfg = load_config(); P = paths(cfg)

### Aperçu

In [6]:
from common import read_jsonl
d = read_jsonl(P["tables"] / "detections_detr.jsonl"); print(len(d), "images ;", d[0])

200 images ; {'image_id': 2261, 'split': 'test', 'detections': [{'category': 'surfboard', 'score': 0.932, 'bbox_norm': [0.4035, 0.5823, 0.6, 0.6971]}, {'category': 'surfboard', 'score': 0.708, 'bbox_norm': [0.4031, 0.5872, 0.5581, 0.699]}, {'category': 'person', 'score': 0.9962, 'bbox_norm': [0.438, 0.352, 0.5963, 0.6331]}]}
